# 🚗 Auto Auction Labor Forecasting
## Portfolio Analysis Notebook

**Author:** [Your Name]  
**Stack:** Python · SQL · SQLite · Claude API  
**Dataset:** 1 year of simulated auto auction operations (2024)

---

This notebook walks through the full labor forecasting pipeline:
1. Data overview & validation
2. Volume patterns by day-of-week and season
3. SQL-driven staffing forecasts
4. Variance analysis (planned vs. actual)
5. AI-generated supervisor briefings

---


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from forecast.forecast_engine import (
    get_week_forecast, get_wow_variance,
    get_monthly_trend, get_anomalies,
    build_forecast_context, forecast_staff_for_volume,
)
from forecast.ai_summary import generate_summary
from datetime import date

# Optional: set your API key here or via environment variable
# import os; os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'

DB_PATH = os.path.abspath('../auction_data.db')
print('DB path:', DB_PATH)
print('Exists:', os.path.exists(DB_PATH))


## 1. Data Overview

In [ ]:
conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("SELECT * FROM daily_operations", conn)
conn.close()

print(f"Rows: {len(df)}")
print(f"Date range: {df['date'].min()} → {df['date'].max()}")
df.head(10)


In [ ]:
df.describe()


## 2. Volume Patterns

### 2a. Average daily volume by day of week

In [ ]:
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday']
dow_avg = (df.groupby('day_of_week')['actual_volume']
             .mean()
             .reindex(dow_order))

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(dow_avg.index, dow_avg.values, color='steelblue', edgecolor='white')
ax.bar_label(bars, fmt='%.0f', padding=3, fontsize=9)
ax.set_title('Average Vehicle Volume by Day of Week', fontweight='bold')
ax.set_ylabel('Avg Vehicles')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()


### 2b. Monthly volume trend

In [ ]:
monthly = get_monthly_trend()

fig, ax1 = plt.subplots(figsize=(11, 4))
ax2 = ax1.twinx()

ax1.bar(monthly['month_name'], monthly['total_volume'],
        color='lightsteelblue', label='Total Volume')
ax2.plot(monthly['month_name'], monthly['avg_daily_staff'],
         color='darkorange', marker='o', linewidth=2, label='Avg Daily Staff')

ax1.set_ylabel('Total Monthly Volume')
ax2.set_ylabel('Avg Daily Staff')
ax1.set_title('Monthly Volume & Staffing Trend', fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.show()


## 3. SQL-Driven 7-Day Staffing Forecast

In [ ]:
forecast_date = date(2024, 4, 1)   # Change this to any Monday in 2024
week = get_week_forecast(forecast_date)

display_cols = ['date','day_of_week','is_sale_day','planned_volume',
                'staff_check_in','staff_detailing','staff_transport',
                'staff_title_admin','staff_lane_support','total_planned_staff']
week[display_cols].style.background_gradient(subset=['total_planned_staff'], cmap='Blues')


### What-If: Custom Volume Input

In [ ]:
custom_volume = 250   # ← change this
is_sale = True        # ← change this

breakdown = forecast_staff_for_volume(custom_volume, is_sale)
print(f"For {custom_volume} vehicles on a {'sale' if is_sale else 'non-sale'} day:")
for role, count in breakdown.items():
    label = role.replace('_',' ').title()
    print(f"  {label:<18} {count} staff")


## 4. Variance Analysis (Planned vs. Actual)

In [ ]:
wow = get_wow_variance(date(2024, 12, 31), weeks=12)

colors = ['#ef553b' if v < 0 else '#00cc96' for v in wow['variance_pct']]
fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.bar(wow['week'], wow['variance_pct'], color=colors)
ax.axhline(0, color='gray', linestyle='--', linewidth=1)
ax.bar_label(bars, fmt='%.1f%%', padding=2, fontsize=8)
ax.set_title('Weekly Staffing Variance % (Actual vs Planned)', fontweight='bold')
ax.set_ylabel('Variance %')
ax.set_xlabel('Week')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


### Anomaly Log — High-Variance Days

In [ ]:
anomalies = get_anomalies(threshold_pct=15)
anomalies.style.applymap(
    lambda v: 'color: red' if v == 'Understaffed' else 'color: green',
    subset=['status']
)


## 5. AI Supervisor Briefing

Uses the Claude API to turn forecast data into a plain-English daily briefing.

In [ ]:
ctx = build_forecast_context(date(2024, 4, 2))   # Tuesday = sale day

import json
print("=== Forecast Context Passed to Claude ===")
print(json.dumps(ctx, indent=2))


In [ ]:
# Requires ANTHROPIC_API_KEY environment variable
summary = generate_summary(ctx)
print("=== AI Supervisor Briefing ===")
print(summary)


---
## Summary

This project demonstrates:

| Skill | Implementation |
|---|---|
| **SQL** | SQLite queries for staffing forecasts, variance analysis, UPLH |
| **Python / pandas** | Data generation, transformation, forecast engine |
| **AI Integration** | Claude API for natural language summaries |
| **Data Visualization** | matplotlib charts for operational patterns |
| **Business Logic** | Labor standards, ceiling staffing calculations, anomaly detection |

The same pipeline powers an interactive Streamlit dashboard — run `streamlit run app/streamlit_app.py` to see it live.
